# Lab 01 — CORTEX.COMPLETE Basics

`SNOWFLAKE.CORTEX.COMPLETE` is the foundational function for calling LLMs inside Snowflake SQL.
This lab covers the core patterns you'll use in every other lab.

| Item | Detail |
|---|---|
| **Function** | `SNOWFLAKE.CORTEX.COMPLETE` |
| **Models** | `claude-haiku-4-5`, `mistral-large2`, `llama3.1-70b` |
| **Exam Domain** | 1.0 Gen AI Overview, 2.0 Gen AI & LLM Functions |
| **You'll learn** | Simple prompts, system/user messages, options, prompt patterns, model comparison |


## Step 1 — Environment Setup

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Basic Completion

The simplest pattern: model name + prompt string.

In [ ]:
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'claude-haiku-4-5',
    'Explain data warehousing in exactly three sentences.'
) AS response;

## Step 3 — System + User Message Pattern

Use the structured message array for role-based prompting. The `system` message sets behavior; the `user` message is the question.

In [ ]:
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'claude-haiku-4-5',
    [
        {'role': 'system', 'content': 'You are a concise Snowflake expert. Answer in bullet points.'},
        {'role': 'user', 'content': 'What are the key benefits of Snowflake Cortex AI functions?'}
    ],
    {'temperature': 0.3, 'max_tokens': 500}
):choices[0]:messages::STRING AS response;

## Step 4 — COMPLETE Options Reference

The third argument controls LLM behavior:

| Option | Type | Description |
|---|---|---|
| `temperature` | FLOAT 0.0–1.0 | Controls randomness. 0 = deterministic. |
| `max_tokens` | INT | Maximum output length in tokens. |
| `top_p` | FLOAT 0.0–1.0 | Nucleus sampling threshold. |
| `guardrails` | BOOLEAN | Enable Cortex Guard safety screening. |
| `response_format` | OBJECT | Force output format, e.g. `{type: 'json'}`. |


In [ ]:
SELECT
    'deterministic' AS mode,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-haiku-4-5',
        [{'role': 'user', 'content': 'Name 3 Snowflake features.'}],
        {'temperature': 0.0, 'max_tokens': 100}
    ):choices[0]:messages::STRING AS response
UNION ALL
SELECT
    'creative',
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-haiku-4-5',
        [{'role': 'user', 'content': 'Name 3 Snowflake features.'}],
        {'temperature': 0.9, 'max_tokens': 100}
    ):choices[0]:messages::STRING;

## Step 5 — Prompt Engineering Patterns

Five patterns you'll use constantly: zero-shot, few-shot, chain-of-thought, extraction, summarization.

In [ ]:
CREATE OR REPLACE TABLE prompt_experiments (
    experiment_id INT AUTOINCREMENT,
    pattern_name  VARCHAR(100),
    prompt_text   TEXT
);

INSERT INTO prompt_experiments (pattern_name, prompt_text) VALUES
('zero_shot', 'Classify this text as POSITIVE, NEGATIVE, or NEUTRAL: The new feature update broke my workflow completely.'),
('few_shot', 'Classify sentiment. Examples:\nInput: Love it! -> POSITIVE\nInput: Terrible. -> NEGATIVE\nInput: It is okay. -> NEUTRAL\n\nInput: The new feature update broke my workflow completely. ->'),
('chain_of_thought', 'Analyze the sentiment step by step:\n1. Identify key phrases\n2. Determine emotional tone\n3. Give final classification\n\nText: The new feature update broke my workflow completely.'),
('extraction', 'Extract the following from this support ticket in JSON format: {issue_type, severity, product_area}.\nTicket: The dashboard export feature has been timing out for the past 3 days. This is blocking our weekly executive report.'),
('summarization', 'Summarize the key technical decisions in this meeting note in exactly 3 bullet points:\nWe discussed migrating from PostgreSQL to Snowflake. Team agreed on a phased approach starting with analytics workloads. Data engineering will own the pipeline migration while the BI team handles dashboard conversion. Target completion is Q2.');

In [ ]:
SELECT
    pattern_name,
    SNOWFLAKE.CORTEX.COMPLETE('claude-haiku-4-5', prompt_text) AS llm_response
FROM prompt_experiments
ORDER BY experiment_id;

## Step 6 — Compare Models

Different models have different strengths. Compare on your specific task.

In [ ]:
SELECT
    'claude-haiku-4-5' AS model,
    SNOWFLAKE.CORTEX.COMPLETE('claude-haiku-4-5', 'In one sentence, what makes Snowflake unique?') AS response
UNION ALL
SELECT
    'mistral-large2',
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', 'In one sentence, what makes Snowflake unique?')
UNION ALL
SELECT
    'llama3.1-70b',
    SNOWFLAKE.CORTEX.COMPLETE('llama3.1-70b', 'In one sentence, what makes Snowflake unique?');

## Key Takeaways

- `COMPLETE` supports both simple string prompts and structured message arrays.
- Use `temperature` (0.0–1.0) to control creativity; lower = more deterministic.
- Five key patterns: zero-shot, few-shot, chain-of-thought, extraction, summarization.
- Apply to table columns with SELECT for batch processing at scale.
- Compare models on your specific task — accuracy, latency, and cost vary.
